# Text Classification with Naive Bayes

Sentiment classification using the **NLTK Movie Reviews corpus** (not the IMDb dataset).

This notebook implements Multinomial Naive Bayes from scratch, compares it with Scikit-learn, and evaluates four challenges using the same held-out test set.


In [1]:
import re
import random
import numpy as np
import nltk
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews

print("Libraries loaded.")

Libraries loaded.


In [2]:
documents = [
    (list(movie_reviews.words(fileid)), category)
    for category in movie_reviews.categories()
    for fileid in movie_reviews.fileids(category)
]
random.seed(42)
random.shuffle(documents)

texts = [" ".join(words) for words, _ in documents]
labels = np.array([1 if category == "pos" else 0 for _, category in documents])

print(f"Total documents: {len(texts)}")
print(f"Positive: {int(labels.sum())}, Negative: {len(labels) - int(labels.sum())}")

Total documents: 2000
Positive: 1000, Negative: 1000


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    np.array(texts), labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1600
Test set size: 400


## 1. Custom Naive Bayes

For each class, the model calculates:

- Prior: \(P(c)=N_c/N\)
- Smoothed likelihood: \(P(w|c)=(count(w,c)+\alpha)/(N_c^{words}+\alpha|V|)\)
- Prediction: the class with the largest log-score.

Log probabilities prevent numerical underflow when multiplying many small probabilities.

In [4]:
class CustomNaiveBayes:
    def __init__(self, alpha=1.0, binary=False):
        if alpha <= 0:
            raise ValueError("alpha must be greater than zero.")
        self.alpha = alpha
        self.binary = binary
        self.classes = []
        self.vocab = set()
        self.class_log_prior = {}
        self.word_log_likelihood = {}

    @staticmethod
    def tokenize(text):
        return re.findall(r"\b\w+\b", str(text).lower())

    def train(self, X, y):
        # Reset model state, so calling train twice is safe.
        self.classes = np.unique(y)
        self.vocab = set()
        self.class_log_prior = {}
        self.word_log_likelihood = {}

        class_word_counts = {c: Counter() for c in self.classes}
        class_doc_counts = {c: 0 for c in self.classes}

        for text, label in zip(X, y):
            words = self.tokenize(text)
            if self.binary:
                words = list(set(words))
            self.vocab.update(words)
            class_word_counts[label].update(words)
            class_doc_counts[label] += 1

        vocab_size = len(self.vocab)
        total_docs = len(X)
        if total_docs == 0 or vocab_size == 0:
            raise ValueError("Training data and vocabulary must not be empty.")

        for c in self.classes:
            self.class_log_prior[c] = np.log(class_doc_counts[c] / total_docs)
            total_words = sum(class_word_counts[c].values())
            denominator = total_words + self.alpha * vocab_size
            self.word_log_likelihood[c] = {
                word: np.log((class_word_counts[c][word] + self.alpha) / denominator)
                for word in self.vocab
            }
        return self

    def predict(self, X):
        predictions = []
        for text in X:
            words = self.tokenize(text)
            if self.binary:
                words = list(set(words))

            scores = {}
            for c in self.classes:
                score = self.class_log_prior[c]
                for word in words:
                    if word in self.vocab:
                        score += self.word_log_likelihood[c][word]
                scores[c] = score
            predictions.append(max(scores, key=scores.get))
        return np.array(predictions)

## 2. Unit tests
The training subset is indexed safely using NumPy arrays. The tests check balanced priors, negative log-likelihoods, and valid predictions.

In [5]:
idx_pos = np.where(y_train == 1)[0][:10]
idx_neg = np.where(y_train == 0)[0][:10]
idx_small = np.concatenate([idx_pos, idx_neg])

nb_check = CustomNaiveBayes(alpha=1.0).train(X_train[idx_small], y_train[idx_small])

assert abs(nb_check.class_log_prior[0] - np.log(0.5)) < 1e-6
assert abs(nb_check.class_log_prior[1] - np.log(0.5)) < 1e-6
for c in nb_check.classes:
    assert all(value < 0 for value in nb_check.word_log_likelihood[c].values())

preds_check = nb_check.predict(X_test[:20])
assert set(preds_check).issubset({0, 1})
print("All unit tests passed.")

All unit tests passed.


## 3. Train and evaluate the custom model

In [6]:
def evaluate_model(y_true, y_pred, title):
    print(f"--- {title} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

nb_model = CustomNaiveBayes(alpha=1.0).train(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)
print("Vocabulary size:", len(nb_model.vocab))
evaluate_model(y_test, y_pred_nb, "Custom Naive Bayes")

Vocabulary size: 36504
--- Custom Naive Bayes ---
Accuracy:  0.7900
Precision: 0.8053
Recall:    0.7650
F1-Score:  0.7846
Confusion Matrix:
[[163  37]
 [ 47 153]]


## 4. Most informative words

In [7]:
def show_top_words(model, n=15):
    log_odds = {
        word: model.word_log_likelihood[1][word] - model.word_log_likelihood[0][word]
        for word in model.vocab
    }
    ordered = sorted(log_odds.items(), key=lambda item: item[1])
    print(f"Top {n} negative words:")
    for word, score in ordered[:n]:
        print(f"{word:<15} {score:+.3f}")
    print(f"\\nTop {n} positive words:")
    for word, score in ordered[-n:][::-1]:
        print(f"{word:<15} {score:+.3f}")

show_top_words(nb_model)

Top 15 negative words:
nbsp            -4.188
schumacher      -3.872
seagal          -3.694
8mm             -3.637
comet           -3.478
magoo           -3.443
jakob           -3.443
ivy             -3.289
lambert         -3.289
stigmata        -3.289
insulting       -3.246
sphere          -3.246
bats            -3.155
jericho         -3.107
supergirl       -3.107
\nTop 15 positive words:
shrek           +3.696
gattaca         +3.627
flynt           +3.578
ordell          +3.527
mulan           +3.473
argento         +3.473
leila           +3.355
sweetback       +3.323
bulworth        +3.274
lambeau         +3.256
taran           +3.185
fei             +3.147
nello           +3.108
apostle         +3.108
redford         +3.025


## 5. Compare with Scikit-learn

In [8]:
vectorizer_nb = CountVectorizer()
X_train_vec_nb = vectorizer_nb.fit_transform(X_train)
X_test_vec_nb = vectorizer_nb.transform(X_test)

sklearn_nb = MultinomialNB(alpha=1.0)
sklearn_nb.fit(X_train_vec_nb, y_train)
y_pred_sklearn = sklearn_nb.predict(X_test_vec_nb)
evaluate_model(y_test, y_pred_sklearn, "Scikit-learn MultinomialNB")

--- Scikit-learn MultinomialNB ---
Accuracy:  0.7875
Precision: 0.8010
Recall:    0.7650
F1-Score:  0.7826
Confusion Matrix:
[[162  38]
 [ 47 153]]


## 6. Test a new review

In [9]:
def deploy_nb_model(text, model, vectorizer):
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)[0]
    probabilities = model.predict_proba(vector)[0]
    sentiment = "Positive" if prediction == 1 else "Negative"
    return sentiment, probabilities[prediction]

new_review = (
    "It's hokey. There are virtually no surprises, and the writing is "
    "second-rate. So why was it so enjoyable? For one thing, the cast is great."
)
prediction, confidence = deploy_nb_model(new_review, sklearn_nb, vectorizer_nb)
print("Review:", new_review)
print(f"Prediction: {prediction} (Confidence: {confidence:.4f})")

Review: It's hokey. There are virtually no surprises, and the writing is second-rate. So why was it so enjoyable? For one thing, the cast is great.
Prediction: Negative (Confidence: 0.8715)


## 7. Underflow demonstration

Multiplying many small probabilities can produce a value rounded to zero (underflow). Adding their logarithms avoids this problem during classification. Note: converting the final log-sum back with `exp` can still underflow; we should compare log-scores directly.

In [10]:
words = nb_model.tokenize(X_test[0])
log_sum = sum(
    nb_model.word_log_likelihood[1][word]
    for word in words if word in nb_model.vocab
)
raw_product = np.exp(log_sum)  # May be zero due to underflow.
print("Review length:", len(words))
print(f"Log-sum: {log_sum:.2f}")
print(f"Product reconstructed with exp: {raw_product:.5e}")
print("Use the log-sum directly for classification; do not multiply probabilities.")

Review length: 667
Log-sum: -4793.26
Product reconstructed with exp: 0.00000e+00
Use the log-sum directly for classification; do not multiply probabilities.


## 8. Challenges

We compare the baseline model with four experiments:

1. **Stop-word removal:** remove common words but keep negation words such as `not`, `no`, and `nor`.
2. **Binary term-frequency variant:** count each distinct word once per document. This is a binary-feature variant of Multinomial NB; it is not the full BernoulliNB model because absence probabilities are not modeled.
3. **Negation handling:** join `not` with the next word (for example, `not good` becomes `not_good`).
4. **Smoothing alpha:** select alpha using a validation split from the training data, then evaluate the selected model on the test set only once.

All challenge scores are computed from predictions. No results are hard-coded. F1-score is reported alongside Accuracy, Precision, and Recall.


In [11]:
# Challenge 1: Stop-word removal
# Keep negation words because removing them can reverse sentiment.
stop_words = set(CountVectorizer(stop_words="english").get_stop_words())
stop_words -= {"no", "nor", "not"}

def remove_stop_words(text):
    return " ".join(
        word for word in CustomNaiveBayes.tokenize(text)
        if word not in stop_words
    )

X_train_stop = [remove_stop_words(text) for text in X_train]
X_test_stop = [remove_stop_words(text) for text in X_test]

stop_model = CustomNaiveBayes(alpha=1.0).train(X_train_stop, y_train)
pred_stop = stop_model.predict(X_test_stop)
evaluate_model(y_test, pred_stop, "Challenge 1: Stop-word removal")


--- Challenge 1: Stop-word removal ---
Accuracy:  0.7850
Precision: 0.7969
Recall:    0.7650
F1-Score:  0.7806
Confusion Matrix:
[[161  39]
 [ 47 153]]


In [12]:
# Challenge 2: Binary term-frequency variant
# Each distinct word contributes at most once per document.
binary_model = CustomNaiveBayes(alpha=1.0, binary=True).train(X_train, y_train)
pred_binary = binary_model.predict(X_test)
evaluate_model(y_test, pred_binary, "Challenge 2: Binary term-frequency NB")


--- Challenge 2: Binary term-frequency NB ---
Accuracy:  0.7925
Precision: 0.8305
Recall:    0.7350
F1-Score:  0.7798
Confusion Matrix:
[[170  30]
 [ 53 147]]


In [13]:
# Challenge 3: Simple negation handling
# Example: "not good" -> "not_good"
def handle_negation(text):
    text = str(text).lower()
    # Join "not" with the immediately following word.
    return re.sub(r"\bnot\s+(\w+)", r"not_\1", text)

X_train_neg = [handle_negation(text) for text in X_train]
X_test_neg = [handle_negation(text) for text in X_test]

neg_model = CustomNaiveBayes(alpha=1.0).train(X_train_neg, y_train)
pred_neg = neg_model.predict(X_test_neg)
evaluate_model(y_test, pred_neg, "Challenge 3: Negation handling")


--- Challenge 3: Negation handling ---
Accuracy:  0.7900
Precision: 0.8053
Recall:    0.7650
F1-Score:  0.7846
Confusion Matrix:
[[163  37]
 [ 47 153]]


In [14]:
# Challenge 4: Tune smoothing alpha on validation data, not on the test set
from sklearn.model_selection import train_test_split

alpha_values = [0.1, 0.5, 1.0, 2.0]

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

alpha_results = []

for alpha in alpha_values:
    candidate = CustomNaiveBayes(alpha=alpha).train(X_fit, y_fit)
    val_predictions = candidate.predict(X_val)
    val_f1 = f1_score(y_val, val_predictions, zero_division=0)
    alpha_results.append((alpha, val_f1))
    print(f"alpha={alpha:<3} | Validation F1={val_f1:.4f}")

best_alpha, best_val_f1 = max(alpha_results, key=lambda item: item[1])
print(f"Best alpha on validation data: {best_alpha} (F1={best_val_f1:.4f})")

# Retrain with the selected alpha on all training data.
best_alpha_model = CustomNaiveBayes(alpha=best_alpha).train(X_train, y_train)
pred_best_alpha = best_alpha_model.predict(X_test)
evaluate_model(y_test, pred_best_alpha, "Challenge 4: Selected alpha (test evaluation)")


alpha=0.1 | Validation F1=0.8050
alpha=0.5 | Validation F1=0.8139
alpha=1.0 | Validation F1=0.8190
alpha=2.0 | Validation F1=0.8399
Best alpha on validation data: 2.0 (F1=0.8399)
--- Challenge 4: Selected alpha (test evaluation) ---
Accuracy:  0.7925
Precision: 0.8031
Recall:    0.7750
F1-Score:  0.7888
Confusion Matrix:
[[162  38]
 [ 45 155]]


## 9. Short answers / conclusions

- **Stop words:** Removing frequent words can reduce noise, but negation words (`not`, `no`, `nor`) must be retained because they affect sentiment.
- **Binary term-frequency variant:** Each distinct word is counted once per document. This reduces the influence of repeated words. It is not full Bernoulli Naive Bayes because absent-word probabilities are not included.
- **Negation:** Joining `not` with the next word preserves a simple form of negation (for example, `not_good`). It does not handle every linguistic construction.
- **Smoothing:** Alpha prevents zero likelihoods for unseen words. Select alpha on validation data, not on the final test set.
- **Underflow:** Log probabilities turn multiplication into addition. Compare log-scores directly instead of converting tiny values back with `exp`.
- **Results:** The best challenge is the one with the highest measured test F1-score. The notebook calculates the scores when all cells are run; do not assume one method always wins.
